# Curvilinear Wave Equation

Add reference-metric terms to the scalar-wave right-hand side and run the curvilinear generated project.

Navigation: [Index](../index.ipynb) | Previous: [Basis Transforms](../4-curvilinear/basis_transforms.ipynb) | Next: [Curvilinear Boundary Conditions](../4-curvilinear/curvilinear_boundary_conditions.ipynb)

## Why This Matters
In spherical or cylindrical-like coordinates the wave operator includes metric and Christoffel terms, not only Cartesian second derivatives.

## What You Will See
You will see the spherical right-hand side, residual checks, generated curvilinear files, build evidence, and diagnostics.

Prerequisite bridge: this notebook follows [Basis Transforms](../4-curvilinear/basis_transforms.ipynb). Terms are defined here before they are used.

## Spherical Right-Hand Side
The residual verifies that the generated spherical expression contains the expected radial, angular, and Christoffel terms.

In [1]:
import sympy as sp

import nrpy.grid as grid
import nrpy.params as par
from nrpy.equations.wave_equation.WaveEquation_RHSs import WaveEquation_RHSs
from nrpy.equations.wave_equation.WaveEquationCurvilinear_RHSs import WaveEquationCurvilinear_RHSs

for name in ["uu", "vv", "uu_rhs", "vv_rhs"]:
    grid.glb_gridfcs_dict.pop(name, None)
spherical_rhs = WaveEquationCurvilinear_RHSs(CoordSystem="Spherical", enable_rfm_precompute=False)
symbols = {symbol.name: symbol for symbol in spherical_rhs.vv_rhs.free_symbols}
wavespeed = symbols["wavespeed"]
xx0 = symbols["xx0"]
xx1 = symbols["xx1"]
uu_dD0 = symbols["uu_dD0"]
uu_dD1 = symbols["uu_dD1"]
uu_dDD00 = symbols["uu_dDD00"]
uu_dDD11 = symbols["uu_dDD11"]
uu_dDD22 = symbols["uu_dDD22"]
expected = wavespeed**2 * (
    uu_dDD00
    + uu_dDD11 / xx0**2
    + uu_dDD22 / (xx0**2 * sp.sin(xx1) ** 2)
    + 2 * uu_dD0 / xx0
    + uu_dD1 * sp.cos(xx1) / (xx0**2 * sp.sin(xx1))
)
residual = sp.trigsimp(sp.simplify(spherical_rhs.vv_rhs - expected))
print("uu_rhs =", spherical_rhs.uu_rhs)
print("vv_rhs =", spherical_rhs.vv_rhs)
print("Spherical RHS residual:", residual)
assert residual == 0

for name in ["uu", "vv", "uu_rhs", "vv_rhs"]:
    grid.glb_gridfcs_dict.pop(name, None)
cartesian_rhs = WaveEquation_RHSs()
for name in ["uu", "vv", "uu_rhs", "vv_rhs"]:
    grid.glb_gridfcs_dict.pop(name, None)
curvilinear_cartesian_rhs = WaveEquationCurvilinear_RHSs(CoordSystem="Cartesian", enable_rfm_precompute=False)
print("Cartesian vv_rhs residual:", sp.simplify(curvilinear_cartesian_rhs.vv_rhs - cartesian_rhs.vv_rhs))

Setting up reference_metric[Spherical]...


uu_rhs = vv
vv_rhs = wavespeed**2*(2*uu_dD0/xx0 + uu_dD1*cos(xx1)/(xx0**2*sin(xx1)) + uu_dDD00 + uu_dDD11/xx0**2 + uu_dDD22/(xx0**2*sin(xx1)**2))
Spherical RHS residual: 0
Setting up reference_metric[Cartesian]...
Cartesian vv_rhs residual: 0


## Generate, Build, and Run the Curvilinear Project
The real generator writes coordinate-specific kernels and diagnostics. The parameter file is shortened only after generation so notebook execution remains quick.

In [2]:
from pathlib import Path
import re
import subprocess
import sys
import tempfile

PROJECT_NAME = "wave_equation_curvilinear"
workspace_manager = tempfile.TemporaryDirectory(prefix="nrpy_tutorial_curvi_", dir=Path.cwd())
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / PROJECT_NAME

command = [sys.executable, "-m", "nrpy.examples.wave_equation_curvilinear"]
print("generator command: python -m nrpy.examples.wave_equation_curvilinear")
generate = subprocess.run(
    command,
    cwd=WORKSPACE,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
    timeout=300,
)
print(generate.stdout.replace(str(WORKSPACE), "<workspace>"))
assert PROJECT_DIR.is_dir()

parfile = PROJECT_DIR / "wave_equation_curvilinear.par"
par_text = parfile.read_text(encoding="utf-8")
par_text = par_text.replace("t_final = 8.0", "t_final = 0.2")
par_text = par_text.replace("diagnostics_output_every = 0.5", "diagnostics_output_every = 0.1")
par_text = par_text.replace("output_progress_every = 1", "output_progress_every = 1000000")
parfile.write_text(par_text, encoding="utf-8")
print(f"--- runtime {parfile.name} ---")
print(parfile.read_text(encoding="utf-8", errors="replace"))

for relative_path in [
    "Makefile",
    "BHaH_function_prototypes.h",
    "SinhCylindrical/rhs_eval__rfm__SinhCylindrical.c",
    "diagnostics/diagnostics.c",
    "wave_equation_curvilinear.par",
]:
    assert (PROJECT_DIR / relative_path).exists(), relative_path

print("complete project inventory:")
for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))

build = subprocess.run(
    ["make", "-j2"],
    cwd=PROJECT_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
    timeout=300,
)
print("build completed")
print("compiler output line count:", len(build.stdout.splitlines()))

run = subprocess.run(
    [f"./{PROJECT_NAME}", "2.0"],
    cwd=PROJECT_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
    timeout=120,
)
cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", run.stdout)
print("run output:")
for line in cleaned.replace(str(WORKSPACE), "<workspace>").splitlines():
    if line.strip():
        print(line.rstrip())

print("diagnostics:")
diagnostics = sorted(PROJECT_DIR.glob("out0d-grid*.txt"))
assert diagnostics
for diagnostic in diagnostics:
    print(f"\n--- {diagnostic.name} ---")
    text = diagnostic.read_text(encoding="utf-8", errors="replace")
    print(text)
    assert len(text.strip().splitlines()) >= 2

generator command: python -m nrpy.examples.wave_equation_curvilinear


Setting up reference_metric[SinhCylindrical]...
In 0.028s, worker completed task 'register_CFunction_initial_data'
In 0.027s, worker completed task 'register_CFunction_diagnostics_volume_integration'
In 0.027s, worker completed task 'register_CFunction_diagnostics_nearest_grid_center'
In 0.028s, worker completed task '_register_CFunction_diagnostics'
In 0.029s, worker completed task 'register_CFunction_diagnostics_nearest'
In 0.030s, worker completed task 'register_CFunction_diagnostics_nearest_2d_xy_and_yz_planes'
In 0.029s, worker completed task 'register_CFunction_diagnostics_nearest_1d_y_and_z_axes'
In 0.037s, worker completed task 'register_CFunction__Cart_to_xx_and_nearest_i0i1i2'
In 0.051s, worker completed task 'register_CFunction_xx_to_Cart'
In 0.091s, worker completed task 'register_CFunction_diagnostic_gfs_set'
In 0.133s, worker completed task 'register_CFunction_initial_data_exact'
In 0.157s, worker completed task 'register_CFunction_rhs_eval'
In 0.222s, worker completed ta

build completed
compiler output line count: 51


run output:
It: 0 t=0.000 / 0.2 = 0.00% dt=1/379.8 | t/h=0.00 ETA 0h00m00s
WRITING CHECKPOINT: cd struct size = 168 time=0.000000e+00
FINISHED WRITING CHECKPOINT
diagnostics:

--- out0d-grid00-SinhCylindrical-conv_factor-2.00.txt ---
# time DIAG_RELERROR_UU(2) DIAG_RELERROR_VV(3) DIAG_UNUM(4) DIAG_UEXACT(5)
0.000000000000000e+00 0.000000000000000e+00 0.000000000000000e+00 3.999996149390150e+00 3.999996149390150e+00
1.000400197985105e-01 1.963821815503563e-09 -2.033720488457365e-06 3.996661711363614e+00 3.996661703514882e+00



## Interpreting the Output
The symbolic residuals check the curvilinear wave operator, and the generated project output shows the same operator inside a compiled multi-file code. The runtime parameter file is the one used for the displayed diagnostics.

## Where This Leads
- [Reference-Metric Applications](../4-curvilinear/reference_metric_applications.ipynb)
- [Basis Transforms](../4-curvilinear/basis_transforms.ipynb)
- [Curvilinear Boundary Conditions](../4-curvilinear/curvilinear_boundary_conditions.ipynb)
- [GeneralRFM and Fisheye Coordinates](../4-curvilinear/generalrfm_and_fisheye.ipynb)